# Day 06 — Faithfulness, Hallucination & Answer Relevancy

**Module 2 · The Metric Toolkit**

Today we explore three important metrics for evaluating generated answers:

- **Faithfulness** — Is the answer supported by the provided context?
- **Hallucination** — Does the answer contain unsupported or contradictory information?
- **Answer Relevancy** — Does the answer actually address the user's question?

We will use the same context and evaluate different answers against it.

> **Core idea:** Different metrics answer different evaluation questions.

## 1. Setup

We will use the same judge model from the previous days.

In [31]:
import os

from dotenv import load_dotenv
from deepeval.models import LocalModel
from deepeval.models import OpenAIModel
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(
    max_concurrent=2
)

print("Judge:", judge.get_model_name())
print("Max concurrent:", async_config.max_concurrent)

Judge: gpt-4.1-mini
Max concurrent: 2


## 2. Create Our Context

Imagine that our AI application retrieved these two pieces of information from a company's knowledge base.

This information will be used to evaluate the generated answers.

In [32]:
from deepeval.test_case import LLMTestCase

context = [
    "The refund window is 30 days from the purchase date.",
    "Digital products are refundable only if never downloaded.",
]

test_cases = [
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "Yes. Refunds are available within 30 days, and digital products "
            "are refundable if they were never downloaded."
        ),
        context=context,
        retrieval_context=context,
    ),
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "Yes. You can get a refund within 30 days. "
            "We will also give you a free physical copy and a lifetime subscription."
        ),
        context=context,
        retrieval_context=context,
    ),
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "The weather in Portugal is lovely this time of year. "
            "Our office has four floors and a great espresso machine."
        ),
        context=context,
        retrieval_context=context,
    ),
]

print(f"Created {len(test_cases)} test cases.")

Created 3 test cases.


## 3. Faithfulness

**Question:**

> Is the generated answer supported by the provided context?

Faithfulness is especially important for applications such as RAG, where the answer is expected to stay grounded in retrieved information.

```text
Retrieval Context
       ↓
Generated Answer
       ↓
Is the answer supported?

In [33]:
from deepeval.metrics import FaithfulnessMetric

faithfulness = FaithfulnessMetric(
    model=judge,
)

print("Faithfulness metric created.")

Faithfulness metric created.


## 4. Hallucination

**Question:**

> Does the answer contain information that is not supported by the context or conflicts with it?

Hallucination evaluation helps us identify when an AI system introduces information that it should not have generated.

```text
Context
   ↓
Answer
   ↓
Unsupported / conflicting information?

In [34]:
from deepeval.metrics import HallucinationMetric

hallucination = HallucinationMetric(
    model=judge,
)

print("Hallucination metric created.")

Hallucination metric created.


## 5. Answer Relevancy

**Question:**

> Does the generated answer actually address the user's question?

Unlike the previous two metrics, Answer Relevancy does not require retrieval context.

It focuses on the relationship between:

```text
User Question
      ↓
Generated Answer
      ↓
Is the answer relevant?

In [35]:
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy = AnswerRelevancyMetric(
    model=judge,
)

print("Answer Relevancy metric created.")

Answer Relevancy metric created.


## 6. Run All Three Metrics

Now we evaluate the same test cases using all three metrics.

In [36]:
from deepeval import evaluate

metrics = [
    faithfulness,
    hallucination,
    answer_relevancy,
]

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
)

for i, result in enumerate(results.test_results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Test Case {i}")

    for metric_result in result.metrics_data:
        print(
            f"{metric_result.name:<20} "
            f"score={metric_result.score:.2f} "
            f"success={metric_result.success}"
        )
        print(f"Reason: {metric_result.reason}")

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Hallucination Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            Can I get a refund on an ebook I never downloaded?                                     │
│  │     Actual Output:    Yes. You can get a refund within 30 days. We will also give you a free physical        │
│  │                       copy and a lifetime subscription.                                                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Faithfulness     │ 0.33  │ 0.50      │ The score is 0.33 because the actual output incorrectly   │
│              │                  │       │           │ claims a free physical copy and a lifetime                │
│              │                  │       │           │ subscription, both of which are not mentioned in the      │
│              │                  │       │           │ retrieval context, leading to clear contradictions.       │
│        PASS  │ Hallucination    │ 0.50  │ 0.50      │ The score is 0.50 because the actual output cor...        │
│        FAIL  │ Answer Relevancy │ 0.33  │ 0.50      │ The score is 0.33 because the response includes           │
│              │                  │       │           │ irrelevant information about free physical copies and     │
│              │                  │       │           │ lifetime subscriptions, which do not address the refund   │
│              │                  │       │           │ eligibility question for an ebook. However, it still      │
│              │                  │       │           │ partially relates to the topic of refunds, just not       │
│              │                  │       │           │ directly answering the user's specific query.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            Can I get a refund on an ebook I never downloaded?                                     │
│  │     Actual Output:    The weather in Portugal is lovely this time of year. Our office has four floors and    │
│  │                       a great espresso machine.                                                              │
│  └── Metrics                                              

⚠ WARNING: No hyperparameters logged.
» ]8;id=591190;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 15.13s | token cost: 0.0062552 USD)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


Test Case 1
Faithfulness         score=1.00 success=True
Reason: The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!
Hallucination        score=1.00 success=True
Reason: The score is 1.00 because the actual output fully aligns with the context, with no contradictions present.
Answer Relevancy     score=1.00 success=True
Reason: The score is 1.00 because the response fully addresses the question about refund eligibility for an undownloaded ebook without including any irrelevant information.

Test Case 2
Faithfulness         score=1.00 success=True
Reason: The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!
Hallucination        score=0.00 success=False
Reason: The score is 0.00 because the actual output fails to mention key contextual details about the refund window and digital product refundability, r

## 7. How Are These Metrics Different?

The three metrics look at different dimensions of the same answer.

| Metric | Main Question |
|---|---|
| Faithfulness | Is the answer supported by the context? |
| Hallucination | Did the answer introduce unsupported or conflicting information? |
| Answer Relevancy | Does the answer address the user's question? |

This distinction is important.

An answer can be:

- relevant but not grounded,
- grounded but irrelevant,
- or both relevant and grounded.

Therefore, **one metric is rarely enough to describe the quality of an AI application.**

# Day 06 — Key Takeaways

Today we learned three important evaluation dimensions:

### Faithfulness
Checks whether the answer is supported by the provided context.

### Hallucination
Checks for unsupported or conflicting information in the answer.

### Answer Relevancy
Checks whether the answer addresses the user's question.

The important mental model is:

```text
                 AI Answer
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
   Faithfulness  Hallucination  Relevancy
        │            │            │
   Grounded?     Invented?     Useful?